In [1]:
import torch
import torch.nn as nn
# from torch.nn.modules.normalization import LayerNorm
from encoder import encode, decode
import pandas as pd
import time

from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
dinonames = pd.read_csv(
    'dino_train.csv',
    header= None,
    names=['input', 'target']
)
dinonames = dinonames.sample(frac=1).reset_index(drop=True)
dinonames

,input,target
0,*protoc,e
1,*bonat,i
2,*clasmodosauru,s
3,*u,n
4,*euc,e
...,...,...
19905,*zhenyuanlong,$
19906,*tan,y
19907,*heilongjiangosaur,u
19908,*cerasinop,s


In [3]:
class DinoDataset(Dataset):

    def __init__(self, dataframe, maxlen=27):
        self.data = dataframe
        self.maxlen = maxlen

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        word = self.data.iloc[idx]['input']
        target = self.data.iloc[idx]['target']

        # Pad word to 27 chars
        word = word + '_' * (self.maxlen - len(word))

        # Convert chars to token IDs
        x = torch.tensor(encode(word), dtype=torch.long)
        y = torch.tensor(encode(target)[0], dtype=torch.long)

        return x, y

In [4]:
dataset = DinoDataset(dinonames)

trainloader = DataLoader(
    dataset, 
    batch_size=32,
    shuffle=True
)

In [5]:
x, y = next(iter(trainloader))

print(x.shape, y.shape)

torch.Size([32, 27]) torch.Size([32])


In [2]:
class DinoEmbedding(nn.Module):

    def __init__(self, vocab_size=29, dmodel=64):
        super().__init__()

        # E.Shape = [vocab_size, dmodel]
        self.E = nn.Parameter(torch.randn(vocab_size, dmodel))

    def forward(self, x):
        # Input: [B, seqlength]
        # Output: [B, seqlength, dmodel]

        # x.shape = [B, seqlength]
        # Each element of x is a token ID in [0, vocab_size-1]
        #
        # E[x] performs a row lookup:
        # [B, seqlength] -> [B, seqlength, dmodel]
        #
        # Output.shape = [B, seqlength, dmodel]
        
        return self.E[x]

class PositionEncoder(nn.Module):

    def __init__(self, seqlength=27, dmodel=64):
        super().__init__()

        self.seqlength = seqlength
        self.dmodel = dmodel

        # PE.shape = [seqlength, dmodel]
        PE = torch.zeros(seqlength, dmodel)

        for pos in range(seqlength):

            # i = 0, 2, 4, ... 62
            # Even dimensions use sin
            # Odd dimensions use cos
            for i in range(0, dmodel, 2):

                denominator = 10000 ** (i / dmodel)

                PE[pos, i] = torch.sin(
                    torch.tensor(pos / denominator)
                )

                PE[pos, i + 1] = torch.cos(
                    torch.tensor(pos / denominator)
                )

        # PE is part of the model, but is NOT trainable.
        self.register_buffer("PE", PE)


    def forward(self, X):

        # X.shape  = [B, seqlength, dmodel]
        # PE.shape = [   seqlength, dmodel]
        #
        # Decoder will perform:
        #
        # X + PE
        #
        # [B,27,64] + [27,64]
        #       -> [B,27,64]
        #
        # PE broadcasts across the batch dimension.

        return self.PE


class AttentionHead(nn.Module):

    def __init__(self, dmodel=64, headers=4):
        super().__init__()

        self.dmodel = dmodel
        self.dk = dmodel // headers
        self.dq = self.dk
        self.dv = self.dk

        # Wq.shape = [dmodel, dq] = [64,16]
        # Wk.shape = [dmodel, dk] = [64,16]
        # Wv.shape = [dmodel, dv] = [64,16]

        self.Wq = nn.Parameter(torch.randn(self.dmodel, self.dq))
        self.Wk = nn.Parameter(torch.randn(self.dmodel, self.dk))
        self.Wv = nn.Parameter(torch.randn(self.dmodel, self.dv))

    def forward(self, X):

        # X.shape = [B, seqlength, dmodel]
        #
        # [B,n,64] x [64,16] -> [B,n,16]

        Q = torch.matmul(X, self.Wq)
        K = torch.matmul(X, self.Wk)
        V = torch.matmul(X, self.Wv)

        # Q.shape = [B,n,16]
        # K.T     = [B,16,n]
        #
        # [B,n,16] x [B,16,n]
        #       -> [B,n,n]

        A = torch.matmul(Q, K.transpose(-2, -1))

        # Scaled dot-product attention
        A = A / (self.dk ** 0.5)

         # ----------------------------------------
        # CAUSAL MASK
        # ----------------------------------------

        n = X.shape[1]

        # mask.shape = [n,n]
        #
        # 1 0 0 0
        # 1 1 0 0
        # 1 1 1 0
        # 1 1 1 1

        mask = torch.tril(
            torch.ones(n, n, device=X.device)
        )

        # A.shape    = [B,n,n]
        # mask.shape = [  n,n]
        #
        # mask broadcasts across batch dimension.
        #
        # Future positions become -infinity.

        A = A.masked_fill(
            mask == 0,
            float('-inf')
        )

        # Softmax converts scores to attention weights.
        #
        # -inf -> probability 0
        #
        # A.shape remains [B,n,n]

        # A.shape remains [B,n,n]
        A = torch.softmax(A, dim=-1)

        # [B,n,n] x [B,n,16]
        #       -> [B,n,16]

        V_attn = torch.matmul(A, V)

        return V_attn
        


class MHA(nn.Module):

    def __init__(self, dmodel=64, headers = 4):
        
        super().__init__()

        self.attention1 = AttentionHead(dmodel, headers)
        self.attention2 = AttentionHead(dmodel, headers)
        self.attention3 = AttentionHead(dmodel, headers)
        self.attention4 = AttentionHead(dmodel, headers)

        self.Wo = nn.Parameter(torch.randn(dmodel, dmodel))
        self.bo = nn.Parameter(torch.randn(dmodel))
        

    def forward(self, X):

        # output of attention layers X1...H4 is [B, seqlength, dk=64/4 = 16]
        X1 = self.attention1(X)
        X2 = self.attention2(X)
        X3 = self.attention3(X)
        X4 = self.attention4(X)

        # concat headers X1,...X4 such that we get [B, seqlenght, dmodel (= dk * headers, 16 * 4 = 64)]
        X = torch.concat((X1, X2, X3, X4), dim=-1)

        # Wo has dimensions [dk * headers (or dmodel), dmodel]
        # Output dimentions [B, seqlength, dmodel]
        X = torch.matmul(X, self.Wo) + self.bo

        return X

class FFN(nn.Module):

    def __init__(self, dmodel=64):

        super().__init__()
        self.dmodel = dmodel

        self.dff = dmodel * 4

        self.W1 = nn.Parameter(torch.randn(self.dmodel, self.dff))
        self.b1 = nn.Parameter(torch.randn(self.dff))
        self.relu = nn.ReLU()

        self.W2 = nn.Parameter(torch.randn(self.dff, self.dmodel))
        self.b2 = nn.Parameter(torch.randn(self.dmodel))
        

    def forward(self, X):

        # X.shape = [B, seqlength, dmodel]
        # W1.shape = [dmodel, dff]
        # b1.shape = [dff]
        # X x W = [B, seqlenght, dff]
        X = torch.matmul(X, self.W1) + self.b1
        X = self.relu(X)
        X = torch.matmul(X, self.W2) + self.b2
        

        return X

        



In [3]:
class Decoder(nn.Module):

    def __init__(self, vocabsize=29, seqlength=27, dmodel = 64):
        super().__init__()

        self.seqlength = seqlength
        self.dmodel = dmodel
        self.vocabsize = vocabsize

        # Initiate the sub Networs
        self.dino_embedding = DinoEmbedding(vocabsize, dmodel)
        self.position_encoder = PositionEncoder(seqlength, dmodel)
        self.mha = MHA(dmodel=dmodel)
        self.norm1 = nn.LayerNorm(dmodel)
        self.ffn = FFN(dmodel=dmodel)
        self.norm2 = nn.LayerNorm(dmodel)
        self.linear = nn.Linear(dmodel, vocabsize)

    def forward(self, X):

        # Input Embedding
        X = self.dino_embedding(X)
        
        # Position Encoding
        X1 = self.position_encoder(X)
        X = X + X1

        # Multi Head Attention
        X2 = self.mha(X)
        X = X2 + X
        X = self.norm1(X)

        # Feed Forward Network
        X3 = self.ffn(X)
        X = X3 + X
        X = self.norm2(X)

        # Linear Projection to vocabulary
        # X has [B, seqlength, dmodel]
        # Output [B, seqlength, vocabsize]
        X = self.linear(X)

        return X
        

In [4]:
model = Decoder()
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

PAD_ID=28


Decoder(
  (dino_embedding): DinoEmbedding()
  (position_encoder): PositionEncoder()
  (mha): MHA(
    (attention1): AttentionHead()
    (attention2): AttentionHead()
    (attention3): AttentionHead()
    (attention4): AttentionHead()
  )
  (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (ffn): FFN(
    (relu): ReLU()
  )
  (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (linear): Linear(in_features=64, out_features=29, bias=True)
)

Total parameters:     53,533
Trainable parameters: 53,533


In [30]:
x, y = next(iter(trainloader))

optimizer.zero_grad()

logits = model(x)

lengths = (x != PAD_ID).sum(dim=1)
last_positions = lengths - 1
batch_idx = torch.arange(x.size(0))

last_logits = logits[batch_idx, last_positions, :]

loss = criterion(last_logits, y)

print("Loss before backward:", loss.item())

loss.backward()
optimizer.step()

print("One training step completed.")

Loss before backward: 3.602141857147217
One training step completed.


In [31]:
epochs = 50

for epoch in range(epochs):

    model.train()
    running_loss = 0.0

    for x, y in trainloader:

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        # [B,27] -> [B,27,29]
        logits = model(x)

        # Find last real character in each prefix
        lengths = (x != PAD_ID).sum(dim=1)
        last_positions = lengths - 1
        batch_idx = torch.arange(x.size(0))

        # [B,27,29] -> [B,29]
        last_logits = logits[
            batch_idx,
            last_positions,
            :
        ]

        # Calculate loss
        loss = criterion(last_logits, y)

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainloader)

    print(
        f"Epoch {epoch+1:2d}/{epochs} "
        f"Loss: {average_loss:.4f}"
    )

Epoch  1/50 Loss: 2.3690
Epoch  2/50 Loss: 2.0759
Epoch  3/50 Loss: 1.9832
Epoch  4/50 Loss: 1.9291
Epoch  5/50 Loss: 1.8996
Epoch  6/50 Loss: 1.8703
Epoch  7/50 Loss: 1.8553
Epoch  8/50 Loss: 1.8316
Epoch  9/50 Loss: 1.8184
Epoch 10/50 Loss: 1.8124
Epoch 11/50 Loss: 1.7875
Epoch 12/50 Loss: 1.7693
Epoch 13/50 Loss: 1.7542
Epoch 14/50 Loss: 1.7473
Epoch 15/50 Loss: 1.7353
Epoch 16/50 Loss: 1.7351
Epoch 17/50 Loss: 1.7227
Epoch 18/50 Loss: 1.7073
Epoch 19/50 Loss: 1.7004
Epoch 20/50 Loss: 1.7023
Epoch 21/50 Loss: 1.6893
Epoch 22/50 Loss: 1.6805
Epoch 23/50 Loss: 1.6778
Epoch 24/50 Loss: 1.6794
Epoch 25/50 Loss: 1.6646
Epoch 26/50 Loss: 1.6506
Epoch 27/50 Loss: 1.6466
Epoch 28/50 Loss: 1.6458
Epoch 29/50 Loss: 1.6359
Epoch 30/50 Loss: 1.6320
Epoch 31/50 Loss: 1.6266
Epoch 32/50 Loss: 1.6273
Epoch 33/50 Loss: 1.6264
Epoch 34/50 Loss: 1.6118
Epoch 35/50 Loss: 1.6049
Epoch 36/50 Loss: 1.5999
Epoch 37/50 Loss: 1.6024
Epoch 38/50 Loss: 1.5944
Epoch 39/50 Loss: 1.5899
Epoch 40/50 Loss: 1.5886


In [5]:
def generate_name(model, seed="Ganga", max_len=27):

    model.eval()

    # Training convention:
    # lowercase + START token
    name = "*" + seed.lower()

    print("Starting: " + name, end="", flush=True)

    with torch.no_grad():

        while len(name) < max_len:

            # ---------------------------------------
            # Prepare input
            # ---------------------------------------

            # Pad to seqlength = 27
            padded_name = name + "_" * (max_len - len(name))

            # Encode characters -> token IDs
            x = torch.tensor(
                encode(padded_name),
                dtype=torch.long
            )

            # Add batch dimension
            #
            # [27] -> [1,27]
            x = x.unsqueeze(0)

            # ---------------------------------------
            # Forward pass
            # ---------------------------------------

            # [1,27] -> [1,27,29]
            logits = model(x)

            # Position of last real character
            last_position = len(name) - 1

            # Get the 29 logits at that position
            #
            # [1,27,29]
            #       ↓
            # [29]
            next_logits = logits[0, last_position, :]

            # ---------------------------------------
            # Greedy decoding
            # ---------------------------------------

            # next_token = torch.argmax(next_logits).item()
            probabilities = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probabilities, 1).item()

            next_char = decode([next_token])

            # ---------------------------------------
            # END token?
            # ---------------------------------------

            if next_char == "$":
                # print(name + "$")
                break

            # Append predicted character
            name = name + next_char

            # print(name)
            # stream only the new character
            print(next_char, end="", flush=True)
            time.sleep(0.12)
            

    print()
    # Remove START token before returning
    return name[1:]



In [61]:
generate_name(model, "loe")

Starting: *loe
*loel
*loell
*loello
*loellol
*loelloli
*loellolid
*loellolido
*loellolidok
*loellolidokl
*loellolidokle
*loellolidokles
*loellolidokles$


'loellolidokles'

In [66]:
generate_name(model, "")

Starting: *
*s
*sa
*sal
*sale
*salen
*saleni
*salenin
*salenina
*saleninan
*saleninan$


'saleninan'

In [67]:
torch.save(model.state_dict(), "dino_gpt.pth")

In [63]:
model.state_dict()

OrderedDict([('dino_embedding.E',
              tensor([[ 0.5792,  1.5167, -0.0285,  ...,  0.5603,  0.1265, -0.5384],
                      [ 1.3362,  0.6531, -0.9447,  ..., -0.9207, -1.3626,  0.5397],
                      [ 0.1587,  0.3618,  0.1970,  ...,  0.6986, -0.4462, -1.5512],
                      ...,
                      [-0.8992, -0.7889,  0.0878,  ...,  0.7716,  1.4954, -1.3787],
                      [-0.5512, -0.3988,  0.3901,  ..., -0.3107,  0.2454, -0.8590],
                      [-0.0794, -1.5007, -1.8172,  ...,  0.9176,  0.1292, -0.2770]])),
             ('position_encoder.PE',
              tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
                        0.0000e+00,  1.0000e+00],
                      [ 8.4147e-01,  5.4030e-01,  6.8156e-01,  ...,  1.0000e+00,
                        1.3335e-04,  1.0000e+00],
                      [ 9.0930e-01, -4.1615e-01,  9.9748e-01,  ...,  1.0000e+00,
                        2.6670e-04,  1.0000e+00],
  

In [6]:
model = Decoder()

model.load_state_dict(
    torch.load("dino_gpt.pth", map_location="cpu")
)

model.eval()

Decoder(
  (dino_embedding): DinoEmbedding()
  (position_encoder): PositionEncoder()
  (mha): MHA(
    (attention1): AttentionHead()
    (attention2): AttentionHead()
    (attention3): AttentionHead()
    (attention4): AttentionHead()
  )
  (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (ffn): FFN(
    (relu): ReLU()
  )
  (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (linear): Linear(in_features=64, out_features=29, bias=True)
)

In [ ]:
while True:
    seed = input("\nSeed: ")

    if seed.lower() == "quit":
        break

    name = generate_name(model, seed=seed)

    print("🦖 DinoGPT:", name)


Seed:  golu


Starting: *goluacesaurus
🦖 DinoGPT: goluacesaurus


In [24]:
import ipywidgets as widgets
from IPython.display import display, clear_output

name_box = widgets.Text(
    placeholder="Type your name...",
    description="Name:"
)

button = widgets.Button(
    description="🦖 MAKE A DINO!"
)

output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        generate_name(model, name_box.value)

button.on_click(on_click)

display(name_box, button, output)

Text(value='', description='Name:', placeholder='Type your name...')

Button(description='🦖 MAKE A DINO!', style=ButtonStyle())

Output()